# pymefpp_partition_data

Notebook to create a partition of the calibration files for later reduction.

The user needs to indicate the following directories:

 - raw_data_dir: directory where the raw data is located
 - reduced_science_dir: directory root where the data reduction arena will be established.
 
 The following is the sequence of events:
 
 - run this notebook to create the data reduction arena
 - run the notebook pymefpp_create_associations.ipynb to create the links between science data and calibration  data.
 - run pymefpp_run.ipynb to actually run the mefpp data reduction engine
 - run pymefpp_solve.ipynb to create a good starting point for a WCS in eacg frame
 - run pymefpp_scamp.ipynb to refine the WCS for a proper combination with swarp
 - run pymefpp_swarp.ipynb to create the coadded frames
 - [NOT YET WRITTEN] run pymefpp:zerpoint.ipynb to calculate the zero points of the frames.

In [ ]:
# importing needed system modules
import glob
import sys
import os
from pathlib import Path
import logging

# here we import the local mefpp_run library
sys.path.insert(0,'/home/fselman/omegacam/mefpp/python')
from mefpp_partition_data import *

In [ ]:
# we set the data reduction directories
# This is in Gluster
raw_data_dir = '/home/lschmidt/fernando/fits/'
# These directories are in the local fast storage
reduced_cal_dir = '/home/fselman/local_storage/omegacam/testing/reduced/calibs/'
reduced_science_dir = '/home/fselman/local_storage/omegacam/testing/reduced/'
log_dir = reduced_science_dir + '/logs/'


try:
    os.makedirs(reduced_cal_dir)
except Exception as e:
    print(e)
    pass

try:
    #os.makedirs(reduced_science_dir)
    os.makedirs(reduced_science_dir+'/logs/')
except Exception as e:
    print(e)
    pass

# we prepar3 the logs
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_dir+"/pymeffpp_partition_data.log"),
                              logging.StreamHandler()
                    ])
    

In [ ]:
# we find all OMEGACAM fits files below the raw data directory
listing = glob.glob(raw_data_dir+'OMEGA.*.fits')

logging.info('%d OmegaCAM frames in %s'%(len(listing), raw_data_dir))

exposurePartition = PartitionsTree()


i_exposure = 0

for file_name in listing:

    i_exposure+= 1
    exposure = Exposure(file_name)
    exposurePartition.insertExposure(exposure)
    biasDict = {k:v for (k,v) in exposurePartition.partList.items() if 'bias/' in k}
    domeDict = {k:v for (k,v) in exposurePartition.partList.items() if 'dome/' in k}
    skyDict = {k:v for (k,v) in exposurePartition.partList.items() if 'sky/' in k}
    stdDict = {k:v for (k,v) in exposurePartition.partList.items() if 'std/' in k}
    sciDict = {k:v for (k,v) in exposurePartition.partList.items() if 'science/' in k}
    if i_exposure%10:
        Total = len(biasDict) + len(domeDict) + \
                len(skyDict) + len(stdDict) +\
                len(sciDict)
        print("BIAS:     %d" % len(biasDict), \
                          "DOME:     %d" % len(domeDict), \
                          "SKY:      %d" % len(skyDict), \
                          "STD:      %d" % len(stdDict), \
                          "SCIENCE:  %d" % len(sciDict), \
                          "Total     %d" % Total, end='\r')


In [ ]:
# we create directory structure to reduce the calibration data

# bad pixel masks BPM are needed but static at this point we create
# a directory to contain them
try:
    os.makedirs(reduced_cal_dir+'bpm')
except:
    pass

Path(reduced_cal_dir+'bpm'+'/bpm.list').touch()
Path(reduced_cal_dir+'bpm'+'/toreduce_bpm').touch()


In [ ]:
# here we create a bias.list file in each bias partition
createFrameList(reduced_cal_dir, 'bias.list', biasDict)

In [ ]:
# here we create a dome.list file in dome partition
createFrameList(reduced_cal_dir, 'dome.list', domeDict)

In [ ]:
# here we create a sky.list file in each sky partition
createFrameList(reduced_cal_dir, 'sky.list', skyDict)

In [ ]:
# here we create a science.list file in each science partition
createFrameList(reduced_science_dir, 'science.list', sciDict)